In [1]:
import os

from langsmith import Client
from qdrant_client import QdrantClient

# Swap OpenAI for Groq/Cohere Langchain wrappers
from langchain_groq import ChatGroq
from langchain_cohere import CohereEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

import cohere
from groq import Groq

c:\Users\Chanreach\Documents\AI-Engineer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Download an example reference data point from LangSmith

In [2]:
client = Client()

In [3]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset-50"
)

In [4]:
dataset

Dataset(name='rag-evaluation-dataset-50', description='Dataset of 50 questions for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('b159b4a1-0569-4956-986c-7236caf3c28c'), created_at=datetime.datetime(2026, 8, 6, 8, 45, 32, 610197, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 8, 6, 8, 45, 32, 610197, tzinfo=TzInfo(0)), example_count=50, session_count=2, last_session_start_time=datetime.datetime(2026, 8, 6, 9, 50, 37, 813372), inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.10.15', 'runtime_version': '3.12.10', 'langchain_version': '1.3.14', 'py_implementation': 'CPython', 'langchain_core_version': '1.5.3'}})

In [5]:
list(client.list_examples(dataset_id=dataset.id, limit=50))[49].outputs

{'ground_truth': 'We have two smartwatches available: the Boean Smart Watch and the KOSPET Military Smart Watch. The KOSPET watch has more sport modes, offering 24 modes, while the Boean watch offers 14.',
 'reference_context_ids': ['B0B2R7HK3C', 'B0B8S534KB'],
 'reference_descriptions': ['Smart Watch, Fitness Tracker with 5ATM Waterproof, Heart Rate Monitor and Blood Oxygen Monitor, Sleep Monitor 1.69-inch HD Color Screen Smartwatch for Men Women Compatible with iPhone & Android 【Customized Your Unique Smart watch】Boean smart watch with 1.69-inch TFT-LCD full touch screen technology makes your operation quick and easy. You can set the watch face to show your personality with the existing images or upload images. The smart watch will vibrate when receiving incoming calls, message notifications from Facebook, Instagram, Twitter, WhatsApp, etc. Never miss any important calls or messages. 【Long Battery & 5ATM Waterproof】It only takes 2 hours to charge and can be used for 8-15 days or 30 d

In [6]:
list(client.list_examples(dataset_id=dataset.id, limit=50))[49].inputs

{'question': 'What smartwatches do you have available and which one has more sport modes?'}

In [7]:
reference_inputs = list(client.list_examples(dataset_id=dataset.id, limit=50))[49].inputs
reference_outputs = list(client.list_examples(dataset_id=dataset.id, limit=50))[49].outputs

## RAG Pipeline

In [8]:
# 1. Initialize the base clients (needed for the pipeline)
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(url="http://localhost:6333")
co_client = cohere.Client()
groq_client = Groq()

# 2. Define all the helper functions
def get_embedding(text, model="embed-english-v3.0"):
    response = co_client.embed(texts=[text], model=model, input_type="search_query")
    return response.embeddings[0]

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )
    
    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_contexts.append(result.payload.get("description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])
        
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_contexts"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question):
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.
Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    # Using the powerful Llama 70B model!
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# 3. Define the main rag_pipeline function
def rag_pipeline(question, top_k=5):
    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    
    # Returning a dictionary instead of just a string so Ragas can read the context!
    return {
        "question": question,
        "answer": answer,
        "retrieved_contexts": retrieved_context["retrieved_contexts"],
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"]
    }

In [9]:
rag_pipeline("can I get some charger ?", top_k=5)

{'question': 'can I get some charger ?',
 'answer': "Based on the available products, I have a few options for you:\n\n1. You can get a Tomoson USB C Cable, 6FT (ID: B0B3RQDR85) which is a 6-foot long cable that supports 60W fast charging and LED lighting.\n2. You can get a USB C to 3.5mm Headphone and Charger Adapter (ID: B0BYCMG5BC) which supports both music playback and charging.\n3. There is no stand-alone charger in the list of available products.\n\nIf you're interested in any of these options, please let me know and I'll be happy to help with the purchase.",
 'retrieved_contexts': ['Tomoson USB C Cable, 6FT Light up USB C to USB C Cable, 60W(3A) LED USB C Charger Cable for iPad Mini 6, iPad Pro 2021/2020, iPad Air 4, MacBook Pro 2020, Samsung Galaxy S23/S22 (Multicolored) ✔Light Up iPhone Charger -- The innovative LED light up iPhone charger cord makes charging more vivid. Especially in the dark, the lighting charging cable easily decorates your room, cool and unique, and also q

## RAGAS Metrics


In [10]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

C:\Users\Chanreach\AppData\Local\Temp\ipykernel_28260\3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\Chanreach\AppData\Local\Temp\ipykernel_28260\3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\Chanreach\AppData\Local\Temp\ipykernel_28260\3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0

In [11]:
ragas_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.1-8b-instant"))
# Switching back to the original model to perfectly match your backend!
ragas_embeddings = LangchainEmbeddingsWrapper(CohereEmbeddings(model="embed-english-v3.0"))

C:\Users\Chanreach\AppData\Local\Temp\ipykernel_28260\954186138.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.1-8b-instant"))
C:\Users\Chanreach\AppData\Local\Temp\ipykernel_28260\954186138.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(CohereEmbeddings(model="embed-english-v3.0"))


In [12]:
reference_inputs 

{'question': 'What smartwatches do you have available and which one has more sport modes?'}

In [13]:
reference_outputs

{'ground_truth': 'We have two smartwatches available: the Boean Smart Watch and the KOSPET Military Smart Watch. The KOSPET watch has more sport modes, offering 24 modes, while the Boean watch offers 14.',
 'reference_context_ids': ['B0B2R7HK3C', 'B0B8S534KB'],
 'reference_descriptions': ['Smart Watch, Fitness Tracker with 5ATM Waterproof, Heart Rate Monitor and Blood Oxygen Monitor, Sleep Monitor 1.69-inch HD Color Screen Smartwatch for Men Women Compatible with iPhone & Android 【Customized Your Unique Smart watch】Boean smart watch with 1.69-inch TFT-LCD full touch screen technology makes your operation quick and easy. You can set the watch face to show your personality with the existing images or upload images. The smart watch will vibrate when receiving incoming calls, message notifications from Facebook, Instagram, Twitter, WhatsApp, etc. Never miss any important calls or messages. 【Long Battery & 5ATM Waterproof】It only takes 2 hours to charge and can be used for 8-15 days or 30 d

In [14]:
result = rag_pipeline(reference_inputs["question"])

In [15]:
result


{'question': 'What smartwatches do you have available and which one has more sport modes?',
 'answer': 'We have two smartwatches available.\n\n1. ID: B0B2R7HK3C, a Smart Watch, Fitness Tracker with 1.69-inch HD Color Screen and 5ATM Waterproof, Heart Rate Monitor and Blood Oxygen Monitor, with Sleep Monitor.\n2. ID: B0B8S534KB, a KOSPET Smart Watches for Men - 5ATM/IP69K Waterproof Fitness Smart Watch for Android iPhones with Heart Rate Blood Pressure.\n\nAs for the number of sport modes, ID: B0B8S534KB (KOSPET Smart Watch) has more sport modes with 24 sport modes such as running, walking, trail running, cycling, climbing, skiing, swimming, basketball, football, and baseball.',
 'retrieved_contexts': ['Smart Watch, Fitness Tracker with 5ATM Waterproof, Heart Rate Monitor and Blood Oxygen Monitor, Sleep Monitor 1.69-inch HD Color Screen Smartwatch for Men Women Compatible with iPhone & Android 【Customized Your Unique Smart watch】Boean smart watch with 1.69-inch TFT-LCD full touch screen

In [16]:
async def ragas_faithfulness(run, example):
    
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        # Change this to plural 'contexts'
        retrieved_contexts=run["retrieved_contexts"] 
    )
    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [17]:
score = await ragas_faithfulness(result, "")
print("Faithfulness Score:", score)

Faithfulness Score: 0.88


In [18]:
async def ragas_response_relevancy(run, example):
    
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_contexts"]
    )
    # Using strictness=1 to save your API rate limit!
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings, strictness=1)
    
    return await scorer.single_turn_ascore(sample)

In [19]:
relevancy_score = await ragas_response_relevancy(result, "")
print("Response Relevancy Score:", relevancy_score)

Response Relevancy Score: 0.641476533279142


In [20]:
async def ragas_context_precision_id_based(run, example):
    
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextPrecision()
    
    return await scorer.single_turn_ascore(sample)

In [21]:
score = await ragas_context_precision_id_based(result, reference_outputs)
print("ID Based Context Precision Score:", score)

ID Based Context Precision Score: 0.4


In [22]:
async def ragas_context_recall_id_based(run, example):
    
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextRecall()
    
    return await scorer.single_turn_ascore(sample)

In [23]:
score = await ragas_context_recall_id_based(result, reference_outputs)
print("ID Based Context Recall Score:", score)

ID Based Context Recall Score: 1.0


In [27]:
from langsmith.evaluation import aevaluate

def run_pipeline(inputs: dict):
    # Run our pipeline for each question
    return rag_pipeline(inputs["question"], top_k=5)

# Evaluate all 50 questions in LangSmith!
experiment_results = await aevaluate(
    run_pipeline,
    data="rag-evaluation-dataset-50",
    evaluators=[
        ragas_faithfulness,
        ragas_response_relevancy,
        ragas_context_precision_id_based,
        ragas_context_recall_id_based
    ],
    experiment_prefix="llama-70b-evaluation",
)

TypeError: 'function' object is not iterable